Import Statements

In [1]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from ipyleaflet import Map, Marker, basemaps, GeomanDrawControl, ImageOverlay, AwesomeIcon,Circle
import ipywidgets as widgets
from ipywidgets import HTML
import numpy as np
import pandas as pd
from scipy.interpolate import griddata

Define Function to get distance between GPS Coordinates

In [2]:
def haversine(coords):
    '''Get input of coords and return dist in miles'''
    R = 3961
    rad_coords = [c*(np.pi/180) for c in coords]
    lat1, lon1, lat2, lon2 = rad_coords
    dlat, dlon = lat2-lat1, lon2-lon1

    a = np.sin((dlat/2))**2 + np.cos((lat1))*np.cos((lat2))*(np.sin((dlon/2))**2)
    c = 2 * (np.arctan2(np.sqrt(a), np.sqrt((1-a))))

    return round(R*c,1)

In [12]:
def get_latlong_data(lat, long):

    # url of API endpoint with lat and long coords
    url = f"https://api.weather.gov/points/{lat},{long}"

    # get response
    response = requests.get(url)

    # check response code
    if response.status_code == 200:
        data = response.json()
        return data
    else:
        print(f"Error: {response.status_code}")
        return None
    
def get_station_data(url):
    # get response
    response = requests.get(url)

    # check response code
    if response.status_code == 200:
        data = response.json()
        return data
    else:
        print(f"Error: {response.status_code}")
        return None

# test out API request function
latitude = "37.410276"
longitude = "-79.054780"

# get data associated with GPS coords
expl_data = get_latlong_data(latitude, longitude)

# get nearby stations
nearby_stations = expl_data["properties"]['observationStations']

station_data = get_station_data(nearby_stations)

# determine stations less than a certain distance from GPS coord
close_stations = []
search_radius = 50
for item in station_data["features"]:
    station_coords = item["geometry"]["coordinates"]
    station = item['properties']["stationIdentifier"]
     
    dist_coords = [
        float(latitude),
        float(longitude),
        station_coords[1],
        station_coords[0]
        ]
    station_dist = haversine(dist_coords)

    if station_dist < search_radius:
        close_stations.append({"Station" : station, "GPS Coords": station_coords, "Distance": station_dist})

stations_df = pd.DataFrame(close_stations)

In [13]:
latitude = "37.410276"
longitude = "-79.054780"
center = (latitude, longitude)
map = Map(center = center, basemap=basemaps.Esri.WorldImagery, zoom = 7)
marker = Marker(location=center, draggable=False)
map.add_layer(marker)

circle = Circle(
    location=center,
    radius=int(search_radius*1609.34),
    color="green",
    fill_color="green",
    fill_opacity=0.2
)

# 3. Add to map
map.add_layer(circle)
for idx, row in stations_df.iterrows():
    title = row["Station"]
    coords = row["GPS Coords"][::-1]

    custom_icon = AwesomeIcon(
    marker_color='red',  # The background color of the teardrop
    icon_color='red',    # The color of the 'star' icon itself
    spin=False
)

    #print(coords, title)
    marker = Marker(location=coords, icon = custom_icon, title = title)
    marker.tooltip = HTML(value=f"Station: {title}")
    marker.popup = HTML(value=f"<b>{title}</b>")
    map.add_layer(marker)

map

Map(center=['37.410276', '-79.054780'], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_ti…